<a href="https://colab.research.google.com/github/pedrosampaiom2007-gif/pandas/blob/main/notebooks/04_selecao_e_agrupamento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# SELEÇÃO E AGRUPAMENTO DE DADOS COM PANDAS
# =============================================================================
# Análise das emissões de gases de efeito estufa no território brasileiro,
# criando tabelas e visualizações para entender os dados e gerar insights.
#
# Fonte: SEEG - Sistema de Estimativa de Emissões e Remoções de Gases de
#        Efeito Estufa, Observatório do Clima (http://seeg.eco.br/download)
#
# Roteiro:
#   1. Leitura dos dados
#   2. Ajustando a base de dados
#   3. Modificando o formato do DataFrame (melt)
#   4. Análise dos gases
#   5. Emissão de gases por setor
#   6. Emissão ao longo dos anos
#   7. Desafio: média de emissão por setor ao longo dos anos
#   8. População dos estados
#   9. Unindo os dados (merge) e emissão per capita
#
# Nomes principais:
#   df_emissoes      -> base original (formato largo: um ano por coluna)
#   df_emissoes_ano  -> base no formato longo (colunas 'Ano' e 'Emissão')
#   df_populacao     -> população por município / estado (IBGE)
#   df_emissao_pop   -> emissão de 2021 + população de cada estado
# =============================================================================

# Conecta o Google Drive, onde estão as planilhas
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =============================================================================
# 1. LEITURA DOS DADOS
# =============================================================================
# pandas: biblioteca para manipular tabelas (DataFrames)
import pandas as pd

In [ ]:
# read_excel() lê um arquivo Excel. sheet_name escolhe a aba da planilha.
CAMINHO_SEEG = "/content/drive/MyDrive/Colab Notebooks/1-SEEG10_GERAL-BR_UF_2022.10.27-FINAL-SITE.xlsx"

df_emissoes = pd.read_excel(CAMINHO_SEEG, sheet_name='GEE Estados')
df_emissoes

In [ ]:
# info() mostra a estrutura do DataFrame: quantidade de linhas e colunas,
# quantidade de valores não nulos e o tipo de dado de cada coluna.
df_emissoes.info()

In [ ]:
# =============================================================================
# 2. AJUSTANDO A BASE DE DADOS
# =============================================================================
# Só interessam as emissões feitas pelos estados que estão no inventário
# nacional. Por isso, vamos ver os valores únicos da coluna
# 'Emissão / Remoção / Bunker'.
#
#   - Remoção -> retirada de gases estufa (o contrário da emissão)
#   - NCI     -> dados não contemplados no inventário nacional
#   - Bunker  -> transporte marítimo e aéreo internacional, ou seja,
#                não são emissões diretas dos estados
COLUNA_TIPO = 'Emissão / Remoção / Bunker'

df_emissoes[COLUNA_TIPO].unique()

In [ ]:
# Primeiro, checamos os dados de remoção.
# Filtro booleano: True nas linhas que são 'Remoção NCI' OU ('|') 'Remoção'
(df_emissoes[COLUNA_TIPO] == 'Remoção NCI') | (df_emissoes[COLUNA_TIPO] == 'Remoção')

In [ ]:
# isin() faz o mesmo filtro de forma mais curta: verifica se cada valor
# está presente na lista informada.
TIPOS_REMOCAO = ['Remoção NCI', 'Remoção']

df_emissoes[df_emissoes[COLUNA_TIPO].isin(TIPOS_REMOCAO)]

In [ ]:
# .loc[linhas, colunas] acessa linhas e colunas ao mesmo tempo.
# Aqui: linhas de remoção + colunas dos anos de 1970 a 2021.
df_emissoes.loc[df_emissoes[COLUNA_TIPO].isin(TIPOS_REMOCAO), 1970:2021]

In [ ]:
# Para saber se TODOS os valores de remoção são negativos, usamos max():
# se o maior valor for <= 0, não existe nenhum valor positivo.
df_emissoes.loc[df_emissoes[COLUNA_TIPO].isin(TIPOS_REMOCAO), 1970:2021].max()

# Conclusão: só há valores negativos, então esses dados são mesmo de remoção.

In [ ]:
# Agora verificamos se algum valor do tipo 'Bunker' está ligado a algum estado.
df_emissoes.loc[df_emissoes[COLUNA_TIPO] == 'Bunker', 'Estado'].unique()

In [ ]:
# Mantemos apenas as linhas de 'Emissão'
df_emissoes = df_emissoes[df_emissoes[COLUNA_TIPO] == 'Emissão']
df_emissoes

In [ ]:
# Conhecendo os setores econômicos da base
df_emissoes['Nível 1 - Setor'].unique()

In [ ]:
# Conhecendo os estados da base
df_emissoes['Estado'].unique()

In [ ]:
# Filtro: apenas os estados da região Sul
ESTADOS_SUL = ['RS', 'SC', 'PR']

df_emissoes[df_emissoes['Estado'].isin(ESTADOS_SUL)]

In [ ]:
# Filtro com duas colunas diferentes: danos à Amazônia
# (setor 'Mudança de Uso da Terra e Floresta' E estado 'AM').
# Use '&' (E) / '|' (OU) quando o filtro envolve colunas diferentes;
# o isin() serve para vários valores de UMA mesma coluna.
filtro_amazonia = (
    (df_emissoes['Nível 1 - Setor'] == 'Mudança de Uso da Terra e Floresta')
    & (df_emissoes['Estado'] == 'AM')
)
df_emissoes[filtro_amazonia]

In [ ]:
# Maior emissão da Agropecuária no Pará em 2021
filtro_agro_pa = (
    (df_emissoes['Nível 1 - Setor'] == 'Agropecuária')
    & (df_emissoes['Estado'] == 'PA')
)
df_emissoes.loc[filtro_agro_pa, 2021].max()

In [ ]:
# Como não sobrou nenhuma informação útil na coluna 'Emissão / Remoção / Bunker'
# (agora ela só tem 'Emissão'), removemos a coluna da base.
df_emissoes = df_emissoes.drop(columns=COLUNA_TIPO)
df_emissoes

In [ ]:
# =============================================================================
# 3. MODIFICANDO O FORMATO DO DATAFRAME (melt)
# =============================================================================
# Para fazer agregações, os valores de emissão precisam estar todos em UMA
# coluna. Então as colunas de cada ano vão virar uma coluna só.
#
# Passo 1: separar os nomes das colunas descritivas (Setor até Produto)...
df_emissoes.loc[:, 'Nível 1 - Setor':'Produto'].columns

In [ ]:
colunas_identificacao = list(df_emissoes.loc[:, 'Nível 1 - Setor':'Produto'].columns)
colunas_identificacao

In [ ]:
# ...e os nomes das colunas de anos (1970 a 2021)
df_emissoes.loc[:, 1970:2021].columns

In [ ]:
colunas_anos = list(df_emissoes.loc[:, 1970:2021].columns)
colunas_anos

In [ ]:
# Passo 2: melt() transforma a tabela LARGA (um ano por coluna) em COMPRIDA.
#
#   Antes:
#   | Setor                 | Gás | Estado | 1970 | 1971 |
#   | Processos Industriais | N2O | SP     | 6210 | ...  |
#
#   Depois:
#   | Setor                 | Gás | Estado | Ano  | Emissão |
#   | Processos Industriais | N2O | SP     | 1970 | 6210    |
#   | Processos Industriais | N2O | SP     | 1971 | ...     |
#
# Parâmetros:
#   id_vars    -> colunas que DESCREVEM o dado (Setor, Gás, Estado...).
#                 Não mudam, só se repetem em cada linha nova.
#   value_vars -> colunas que serão "empilhadas" (os anos)
#   var_name   -> nome da coluna que recebe os antigos títulos (ex: 1970)
#   value_name -> nome da coluna que recebe os números das células
#
# Leitura de uma linha: "Processos Industriais, em SP, emitiu 6.210 t de N2O em 1970."
# Sintaxe: dataframe.melt(id_vars=[...], value_vars=[...], var_name='..', value_name='..')
df_emissoes_ano = df_emissoes.melt(
    id_vars=colunas_identificacao,
    value_vars=colunas_anos,
    var_name='Ano',
    value_name='Emissão',
)
df_emissoes_ano

In [ ]:
# =============================================================================
# 4. ANÁLISE DOS GASES
# =============================================================================
# Emissão TOTAL por tipo de gás, agrupando com groupby()
df_emissoes_ano.groupby('Gás')['Emissão'].sum()

In [ ]:
# .groups mostra quais linhas (índices) caíram em cada grupo
df_emissoes_ano.groupby('Gás').groups

In [ ]:
# get_group() seleciona um grupo específico
df_emissoes_ano.groupby('Gás').get_group('CO2 (t)')

In [ ]:
# Agregações possíveis com groupby():
#   count() contagem | sum() soma    | mean() média    | median() mediana
#   min() mínimo     | max() máximo  | std() desvio-padrão | var() variância
#
# Emissão total por gás, ordenada da maior para a menor
emissao_total_por_gas = (
    df_emissoes_ano.groupby('Gás')[['Emissão']]
    .sum()
    .sort_values('Emissão', ascending=False)
)
emissao_total_por_gas

In [ ]:
# Gráfico de barras horizontais para facilitar a comparação entre os gases
emissao_total_por_gas.plot(kind='barh', figsize=(10, 6));

In [ ]:
# iloc seleciona pela POSIÇÃO: as 9 primeiras linhas são as formas de CO2,
# que são as que mais emitem
emissao_co2 = emissao_total_por_gas.iloc[0:9]
emissao_co2

In [ ]:
# Percentual do CO2 em relação ao total de emissões
percentual_co2 = emissao_co2.sum() / emissao_total_por_gas.sum()

print(f"A emissão de CO2 corresponde a {percentual_co2.iloc[0] * 100:.2f}% "
      "da emissão total dos gases de efeito estufa no Brasil de 1970 até 2021")

In [ ]:
# =============================================================================
# 5. EMISSÃO DE GASES POR SETOR
# =============================================================================
# Objetivos:
#   - descobrir o setor mais poluente para cada gás
#   - descobrir o gás mais poluente para cada setor
#
# Agrupando por 2 colunas -> índice com 2 níveis:
#   nível 0 = 'Gás' | nível 1 = 'Nível 1 - Setor'
emissao_gas_setor = df_emissoes_ano.groupby(['Gás', 'Nível 1 - Setor'])[['Emissão']].sum()
emissao_gas_setor

In [ ]:
# xs() (cross-section) seleciona um valor de um nível do índice.
# Aqui: só as linhas do gás CO2
emissao_gas_setor.xs('CO2 (t)', level=0)

In [ ]:
# max() -> o VALOR da maior emissão de CO2
emissao_gas_setor.xs('CO2 (t)', level=0).max()

In [ ]:
# idxmax() -> o NOME (índice) do setor com a maior emissão de CO2
emissao_gas_setor.xs('CO2 (t)', level=0).idxmax()

In [ ]:
# Para não repetir isso gás por gás, usamos groupby(level=0) + idxmax():
# o setor mais emissor de cada gás de uma vez só
emissao_gas_setor.groupby(level=0).idxmax()

In [ ]:
# O valor máximo de emissão de cada gás
emissao_gas_setor.groupby(level=0).max()

In [ ]:
# Tabela resumo: setor mais emissor + quantidade máxima de emissão, por gás
emissao_maxima = emissao_gas_setor.groupby(level=0).max().values

resumo_setor_mais_emissor = emissao_gas_setor.groupby(level=0).idxmax()
resumo_setor_mais_emissor.insert(1, 'Quantidade máxima de emissão', emissao_maxima)
resumo_setor_mais_emissor

In [ ]:
# swaplevel(0, 1) troca a ordem dos níveis do índice: agora Setor vem antes de Gás
emissao_gas_setor.swaplevel(0, 1)

In [ ]:
# Gás mais emitido em cada setor
# (idxmax retorna o nome do índice; max retornaria o valor)
emissao_gas_setor.swaplevel(0, 1).groupby(level=0).idxmax()

In [ ]:
# =============================================================================
# 6. EMISSÃO AO LONGO DOS ANOS
# =============================================================================
# As emissões aumentaram ou diminuíram ao longo dos anos?
df_emissoes_ano

In [ ]:
# Média de emissão por ano
df_emissoes_ano.groupby('Ano')[['Emissão']].mean().plot(figsize=(10, 6));

In [ ]:
# Ano com a maior média de emissão
df_emissoes_ano.groupby('Ano')[['Emissão']].mean().idxmax()

In [ ]:
# Média de emissão por ano E por gás
df_emissoes_ano.groupby(['Ano', 'Gás'])[['Emissão']].mean()

In [ ]:
# reset_index() devolve 'Ano' e 'Gás' como colunas normais
media_ano_gas = df_emissoes_ano.groupby(['Ano', 'Gás'])[['Emissão']].mean().reset_index()
media_ano_gas

In [ ]:
# pivot_table() resume e reorganiza os dados (formato longo -> largo):
#   index   -> o que vira LINHA (cada ano)
#   columns -> o que vira COLUNA (cada gás)
#   values  -> o que preenche as células (a emissão)
#   aggfunc -> função de agregação (padrão: 'mean')
# Assim fica fácil comparar os gases ao longo dos anos.
pivot_media_gas_ano = media_ano_gas.pivot_table(index='Ano', columns='Gás', values='Emissão')
pivot_media_gas_ano

In [ ]:
# subplots=True -> um gráfico separado para cada gás
pivot_media_gas_ano.plot(subplots=True, figsize=(10, 40));

In [ ]:
# =============================================================================
# 7. DESAFIO: MÉDIA DE EMISSÃO POR SETOR AO LONGO DOS ANOS
# =============================================================================
# A equipe de supervisão quer a média de emissão de cada setor
# ('Nível 1 - Setor') ao longo dos anos, usando SÓ pivot_table (sem groupby),
# com tabela + visualização gráfica.
#
# Selecionando só as colunas necessárias
df_setor_ano = df_emissoes_ano[['Nível 1 - Setor', 'Emissão', 'Ano']]
df_setor_ano

In [ ]:
# Média geral de emissão (todos os setores e anos juntos), para referência
media_geral_emissao = df_setor_ano['Emissão'].mean()
media_geral_emissao

In [ ]:
pivot_media_setor_ano = df_setor_ano.pivot_table(
    index='Ano',
    columns='Nível 1 - Setor',
    values='Emissão',
    aggfunc='mean',
)
pivot_media_setor_ano

In [ ]:
pivot_media_setor_ano.plot(
    subplots=True,
    figsize=(10, 15),
    layout=(5, 1),
    title='Média de Emissão por Setor ao Longo dos Anos',
);

In [ ]:
# =============================================================================
# 8. POPULAÇÃO DOS ESTADOS
# =============================================================================
# Nova base com a população de cada município (Censo IBGE 2022), para
# depois calcular a emissão per capita de cada estado.
#   header=1      -> o cabeçalho está na 2ª linha da planilha
#   skipfooter=34 -> ignora as 34 últimas linhas (notas de rodapé)
CAMINHO_POPULACAO = "/content/drive/MyDrive/OneDrive_4_21-09-2026/POP2022_Municipios.xls"

df_populacao = pd.read_excel(CAMINHO_POPULACAO, header=1, skipfooter=34)
df_populacao

In [ ]:
# Os dados são por município, então agregamos por estado (UF)
df_populacao.groupby('UF').sum(numeric_only=True)

In [ ]:
# A população não foi somada -> a coluna não é numérica.
# Tentando converter para número, aparece o erro: há valores com parênteses.
try:
    df_populacao['POPULAÇÃO'].astype(float)
except ValueError as erro:
    print('Erro ao converter:', erro)

In [ ]:
# str.contains() encontra os textos com '(' (r'' = texto "cru", para a barra funcionar)
populacao_com_parenteses = df_populacao[df_populacao['POPULAÇÃO'].str.contains(r'\(', na=False)]
populacao_com_parenteses

In [ ]:
# Limpeza da coluna de população:
#   1. garante que a coluna inteira é texto
#   2. remove as notas entre parênteses, ex: '(1)', '(12)'
#   3. remove os pontos de milhar, ex: '1.234.567' -> '1234567'
#   4. converte para número (o que não der vira NaN)
df_populacao['POPULAÇÃO'] = df_populacao['POPULAÇÃO'].astype(str)

df_populacao = df_populacao.assign(
    populacao_sem_parenteses=df_populacao['POPULAÇÃO'].str.replace(r'\(\d{1,2}\)', '', regex=True),
    populacao=lambda tabela: tabela['populacao_sem_parenteses'].str.replace(r'\.', '', regex=True),
)
df_populacao['populacao'] = pd.to_numeric(df_populacao['populacao'], errors='coerce')
df_populacao

In [ ]:
# Conferindo as linhas que tinham parênteses depois da limpeza
populacao_com_parenteses = df_populacao[df_populacao['POPULAÇÃO'].str.contains(r'\(', na=False)]
populacao_com_parenteses

In [ ]:
# Convertendo para inteiro
df_populacao = df_populacao.astype({'populacao': 'int64'})

In [ ]:
# Agora sim: população total de cada estado
df_populacao_uf = df_populacao.groupby('UF')['populacao'].sum().reset_index()
df_populacao_uf

In [ ]:
# =============================================================================
# 9. UNINDO OS DADOS (merge) E EMISSÃO PER CAPITA
# =============================================================================
# Emissão total de cada estado em 2021 (ano mais próximo do censo de 2022)
#   df_emissoes_ano['Ano'] == 2021 -> filtro: só as linhas de 2021
#   groupby('Estado')...sum()      -> soma as emissões de cada estado
#   reset_index()                  -> devolve 'Estado' como coluna (precisamos dela no merge)
emissao_estados_2021 = (
    df_emissoes_ano[df_emissoes_ano['Ano'] == 2021]
    .groupby('Estado')[['Emissão']]
    .sum()
    .reset_index()
)
emissao_estados_2021

In [ ]:
# merge() junta duas tabelas por uma coluna em comum (como PROCV no Excel / JOIN no SQL)
#   left_on='Estado' -> coluna da tabela da esquerda (emissões)
#   right_on='UF'    -> coluna da tabela da direita (população)
#   (nomes diferentes, mas mesmo conteúdo: 'SP', 'RJ'...; se fossem iguais, bastaria on='Estado')
# Por padrão o merge é 'inner': só ficam os estados que existem nas DUAS tabelas.
df_emissao_pop = pd.merge(emissao_estados_2021, df_populacao_uf, left_on='Estado', right_on='UF')
df_emissao_pop

In [ ]:
# A emissão aumenta conforme o tamanho da população?
# Gráfico de dispersão: cada ponto é um estado.
# Se os pontos sobem da esquerda para a direita -> mais gente = mais emissão.
df_emissao_pop.plot(x='populacao', y='Emissão', kind='scatter', figsize=(8, 6));

In [ ]:
# plotly.express: gráficos interativos (passe o mouse para ver os valores)
import plotly.express as px

In [ ]:
# Scatter interativo com a sigla de cada estado no lugar do ponto
#   text='Estado' -> escreve a sigla do estado
#   opacity=0     -> esconde a bolinha, deixando só a sigla
px.scatter(data_frame=df_emissao_pop, x='populacao', y='Emissão', text='Estado', opacity=0)

In [ ]:
# Emissão per capita = emissão do estado / população do estado
# (quanto cada pessoa "emite", em média). Ordenado do maior para o menor.
df_emissao_pop = df_emissao_pop.assign(
    emissao_per_capita=df_emissao_pop['Emissão'] / df_emissao_pop['populacao']
).sort_values('emissao_per_capita', ascending=False)

df_emissao_pop

In [ ]:
# EXTRA: o Plotly pode interpretar 'emissao_per_capita' como texto.
# pd.to_numeric() garante que a coluna seja numérica antes de plotar.
df_emissao_pop['emissao_per_capita'] = pd.to_numeric(df_emissao_pop['emissao_per_capita'])

# Gráfico de barras: um estado por barra, altura = emissão per capita
px.bar(data_frame=df_emissao_pop, x='Estado', y='emissao_per_capita')